# Round 1 Analysis: EMERALDS & TOMATOES

We have two days of historical order-book snapshots and trade logs.  
**Goal:** understand the micro-structure of each product and design a profitable trading strategy.

| Product | Typical Mid | Spread | Volatility | Character |
|---------|------------|--------|------------|-----------|
| EMERALDS | 10 000 | 16 | ~0.7 stdev | Stable, mean-reverting |
| TOMATOES | ~5 000 | 13 | ~10-15 stdev | Trending, momentum |

In [ ]:
import csv, json, statistics
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 4)})

# ── loaders ──
def load_prices(path):
    with open(path) as f:
        return list(csv.DictReader(f, delimiter=";"))

def load_trades(path):
    with open(path) as f:
        return list(csv.DictReader(f, delimiter=";"))

# load everything
p_d2 = load_prices("TUTORIAL_ROUND_1/prices_round_0_day_-2.csv")
p_d1 = load_prices("TUTORIAL_ROUND_1/prices_round_0_day_-1.csv")
t_d2 = load_trades("TUTORIAL_ROUND_1/trades_round_0_day_-2.csv")
t_d1 = load_trades("TUTORIAL_ROUND_1/trades_round_0_day_-1.csv")

# convenience: split by product
def split(rows, product):
    return [r for r in rows if r.get("product", r.get("symbol")) == product]

print(f"Loaded {len(p_d2)+len(p_d1):,} price ticks, {len(t_d2)+len(t_d1):,} trades")

## 1. EMERALDS: A rock-solid fair value

EMERALDS barely move. The mid-price is pinned at **10 000** with sub-unit variance across both days. This is the signature of a product whose fair value is publicly known -- the only profit opportunity is **providing liquidity** (market-making) inside the wide spread the bots leave open.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Panel 1: Mid-price time series ──
for label, data, color in [("Day -2", p_d2, "steelblue"), ("Day -1", p_d1, "coral")]:
    rows = split(data, "EMERALDS")
    ts = [int(r["timestamp"]) / 1000 for r in rows]
    mids = [float(r["mid_price"]) for r in rows]
    axes[0].plot(ts, mids, color=color, alpha=0.6, linewidth=0.5, label=label)

axes[0].axhline(10000, color="black", ls="--", lw=1, label="Fair = 10000")
axes[0].set_title("EMERALDS Mid-Price")
axes[0].set_xlabel("Timestamp (x1000)")
axes[0].set_ylabel("Mid Price")
axes[0].legend()
axes[0].set_ylim(9994, 10006)

# ── Panel 2: Mid-price histogram ──
all_mids = [float(r["mid_price"]) for r in split(p_d2, "EMERALDS") + split(p_d1, "EMERALDS")]
axes[1].hist(all_mids, bins=50, color="steelblue", edgecolor="white")
axes[1].axvline(10000, color="red", ls="--", lw=1.5)
axes[1].set_title(f"Mid-Price Distribution (stdev={statistics.stdev(all_mids):.2f})")
axes[1].set_xlabel("Mid Price")

# ── Panel 3: Spread distribution ──
spreads = []
for data in [p_d2, p_d1]:
    for r in split(data, "EMERALDS"):
        spreads.append(float(r["ask_price_1"]) - float(r["bid_price_1"]))

spread_counts = defaultdict(int)
for s in spreads:
    spread_counts[s] += 1

bars = sorted(spread_counts.items())
axes[2].bar([str(int(s)) for s, _ in bars], [c for _, c in bars], color=["coral", "steelblue"])
axes[2].set_title("Bid-Ask Spread Distribution")
axes[2].set_xlabel("Spread")
axes[2].set_ylabel("Count (ticks)")
for i, (s, c) in enumerate(bars):
    axes[2].text(i, c + 200, f"{c/len(spreads)*100:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print(f"Mid-price stdev: {statistics.stdev(all_mids):.2f}")
print(f"Spread = 16 in {spread_counts[16.0]/len(spreads)*100:.1f}% of ticks")
print(f"Spread =  8 in {spread_counts[8.0]/len(spreads)*100:.1f}% of ticks (someone quotes at 10000)")

### EMERALDS Order Book Structure

The book is almost always **9992 / 10008** -- a 16-wide spread around fair value. ~3% of the time one side tightens to 10000 (spread = 8). This is the key insight:

> **Nobody is quoting inside the spread.** If we post a bid at 9999 and an ask at 10001, every incoming market order fills against *us* first (we offer a better price than the bots).

In [ ]:
# Visualize the typical EMERALDS order book at a single timestamp
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Snapshot: a typical wide-spread tick and a tight-spread tick
for ax, label, data, ts_target in [
    (axes[0], "Typical Tick (spread=16)", p_d2, 1000),
    (axes[1], "Tight Tick (spread=8)", p_d2, 700),
]:
    row = [r for r in split(data, "EMERALDS") if int(r["timestamp"]) == ts_target][0]

    bid_prices, bid_vols = [], []
    ask_prices, ask_vols = [], []
    for i in range(1, 4):
        bp = row.get(f"bid_price_{i}", "")
        bv = row.get(f"bid_volume_{i}", "")
        ap = row.get(f"ask_price_{i}", "")
        av = row.get(f"ask_volume_{i}", "")
        if bp:
            bid_prices.append(int(float(bp)))
            bid_vols.append(int(bv))
        if ap:
            ask_prices.append(int(float(ap)))
            ask_vols.append(int(av))

    all_prices = bid_prices + ask_prices
    bar_colors = ["forestgreen"] * len(bid_prices) + ["indianred"] * len(ask_prices)
    all_vols = bid_vols + ask_vols

    ax.bar([str(p) for p in all_prices], all_vols, color=bar_colors)
    ax.axvline(len(bid_prices) - 0.5, color="gray", ls=":", lw=1)
    ax.set_title(f"{label}\nt={ts_target}")
    ax.set_xlabel("Price")
    ax.set_ylabel("Volume")

    # Annotate the spread
    spread = min(ask_prices) - max(bid_prices)
    mid_idx = len(bid_prices) - 0.5
    ax.text(mid_idx, max(all_vols) * 0.9, f"spread={spread}",
            ha="center", fontsize=10, color="gray",
            bbox=dict(boxstyle="round", fc="lightyellow"))

axes[0].legend(["Bids (green) | Asks (red)"], loc="upper left", framealpha=0.8)
plt.suptitle("EMERALDS Order Book Snapshots", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### EMERALDS Incoming Trade Flow

The trades file shows us when aggressive orders arrive. Each trade at **10008** means a buy-side market order came in; each trade at **9992** means a sell-side order. Buy/sell flow is roughly balanced -- good for a market maker.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, label, trades in [(axes[0], "Day -2", t_d2), (axes[1], "Day -1", t_d1)]:
    em = split(trades, "EMERALDS")
    prices = [float(t["price"]) for t in em]
    qtys = [int(t["quantity"]) for t in em]

    # colour by side: green = sell-side (at bid), red = buy-side (at ask)
    colors = ["forestgreen" if p <= 9992 else ("indianred" if p >= 10008 else "gold") for p in prices]
    ts = [int(t["timestamp"]) / 1000 for t in em]

    ax.scatter(ts, prices, c=colors, s=[q * 8 for q in qtys], alpha=0.6, edgecolors="black", linewidths=0.3)
    ax.axhline(10000, color="black", ls="--", lw=0.8)
    ax.set_title(f"EMERALDS Trades -- {label}")
    ax.set_xlabel("Timestamp (x1000)")
    ax.set_ylabel("Trade Price")

    buy_vol = sum(q for p, q in zip(prices, qtys) if p >= 10008)
    sell_vol = sum(q for p, q in zip(prices, qtys) if p <= 9992)
    ax.text(0.02, 0.95, f"Buy vol: {buy_vol}\nSell vol: {sell_vol}",
            transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(boxstyle="round", fc="lightyellow"))

plt.suptitle("Roughly balanced flow = safe to market-make", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 2. TOMATOES: A trending market

Unlike EMERALDS, TOMATOES **drift**. Day -2 trended +14 upward; Day -1 dropped -40. The fair value is not fixed -- we need to estimate it dynamically.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, label, data, color in [
    (axes[0], "Day -2", p_d2, "steelblue"),
    (axes[1], "Day -1", p_d1, "coral"),
]:
    rows = split(data, "TOMATOES")
    ts = np.array([int(r["timestamp"]) / 1000 for r in rows])
    mids = np.array([float(r["mid_price"]) for r in rows])

    # EMA-20 and EMA-100
    ema20 = np.zeros_like(mids)
    ema100 = np.zeros_like(mids)
    ema20[0] = ema100[0] = mids[0]
    a20, a100 = 2 / 21, 2 / 101
    for i in range(1, len(mids)):
        ema20[i] = a20 * mids[i] + (1 - a20) * ema20[i - 1]
        ema100[i] = a100 * mids[i] + (1 - a100) * ema100[i - 1]

    ax.plot(ts, mids, color=color, alpha=0.3, linewidth=0.4, label="mid")
    ax.plot(ts, ema20, color="black", linewidth=1.2, label="EMA(20)")
    ax.plot(ts, ema100, color="purple", linewidth=1.2, ls="--", label="EMA(100)")
    ax.set_title(f"TOMATOES -- {label}")
    ax.set_xlabel("Timestamp (x1000)")
    ax.legend(fontsize=8)

    drift = mids[-1] - mids[0]
    ax.text(0.02, 0.05, f"Drift: {drift:+.0f}", transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round", fc="lightyellow"))

axes[0].set_ylabel("Mid Price")
plt.suptitle("TOMATOES trends hard -- need a dynamic fair value", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### TOMATOES Momentum Persistence

Do short-term moves predict the next move? We check: of consecutive 100-tick windows, how often does the direction continue?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, label, data, color in [
    (axes[0], "Day -2", p_d2, "steelblue"),
    (axes[1], "Day -1", p_d1, "coral"),
]:
    rows = split(data, "TOMATOES")
    mids = [float(r["mid_price"]) for r in rows]

    # window means
    w = 100
    n_win = len(mids) // w
    wmeans = [np.mean(mids[i * w : (i + 1) * w]) for i in range(n_win)]
    deltas = np.diff(wmeans)

    # momentum: does the next delta have the same sign?
    same = sum(1 for i in range(len(deltas) - 1) if deltas[i] * deltas[i + 1] > 0)
    total = len(deltas) - 1
    pct = same / total * 100

    ax.bar(range(len(deltas)), deltas, color=[color if d > 0 else "gray" for d in deltas], width=1)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(f"{label}: {pct:.0f}% momentum persistence")
    ax.set_xlabel("100-tick window index")
    ax.set_ylabel("Price change")

plt.suptitle("Consecutive moves tend to continue (>50% = momentum)", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### TOMATOES Trade Flow

More trades, smaller sizes, wider price range. The flow is slightly sell-biased on Day -1 (consistent with the downtrend).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, label, trades, prices in [
    (axes[0], "Day -2", t_d2, p_d2),
    (axes[1], "Day -1", t_d1, p_d1),
]:
    tom_t = split(trades, "TOMATOES")
    tom_p = split(prices, "TOMATOES")

    t_ts = [int(t["timestamp"]) / 1000 for t in tom_t]
    t_prices = [float(t["price"]) for t in tom_t]
    t_qtys = [int(t["quantity"]) for t in tom_t]

    # mid-price as background
    p_ts = [int(r["timestamp"]) / 1000 for r in tom_p]
    p_mids = [float(r["mid_price"]) for r in tom_p]
    ax.plot(p_ts, p_mids, color="gray", alpha=0.3, linewidth=0.4)

    # trades coloured by side relative to mid
    mid_lookup = dict(zip(p_ts, p_mids))
    colors = []
    for tt, tp in zip(t_ts, t_prices):
        nearest_mid = min(mid_lookup.keys(), key=lambda x: abs(x - tt))
        colors.append("indianred" if tp > mid_lookup[nearest_mid] else "forestgreen")

    ax.scatter(t_ts, t_prices, c=colors, s=[q * 10 for q in t_qtys],
               alpha=0.5, edgecolors="black", linewidths=0.3)
    ax.set_title(f"TOMATOES Trades -- {label}")
    ax.set_xlabel("Timestamp (x1000)")

    buy_n = colors.count("indianred")
    sell_n = colors.count("forestgreen")
    ax.text(0.02, 0.95, f"Buy-side: {buy_n}\nSell-side: {sell_n}",
            transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(boxstyle="round", fc="lightyellow"))

axes[0].set_ylabel("Price")
plt.tight_layout()
plt.show()

---
## 3. Strategy Design

### EMERALDS: Market-Make at 9999 / 10001

The fair value is known (10000) and never moves. The bots leave a 16-wide gap. We step in front of them:

1. **Snipe** -- if any bot quotes at or through fair value (ask <= 10000 or bid >= 10000), take it immediately.
2. **Post** -- bid at 9999, ask at 10001 to capture incoming flow for a 2-per-unit profit.
3. **Lean** -- shift quotes against our inventory so we naturally flatten.

### TOMATOES: EMA Market-Making with Trend Bias

Fair value drifts, so we track it with **EMA(20)** and market-make around it:

1. **Snipe** -- take anything more than 1 from our fair-value estimate.
2. **Post** -- bid at FV - 5, ask at FV + 5 (the spread that maximises PnL in backtests).
3. **Lean** -- position-aware skew to flatten inventory.
4. **Trend** -- compare EMA(20) vs EMA(100); if fast > slow (uptrend), shift quotes up slightly.

---
## 4. Backtest

We replay the trade-flow data against our strategy. When an aggressive order arrives, if our posted quote is better than the existing book, the order fills against us instead.

> **Caveat:** this is a lower bound. In the real exchange our orders also sit in the book and attract *new* flow we cannot see in the historical data. The actual PnL should be higher.

In [ ]:
def backtest_emeralds(trades, pos_limit=20):
    """Replay EMERALDS trade flow against our market-making strategy."""
    FAIR = 10000
    em = sorted(split(trades, "EMERALDS"), key=lambda t: int(t["timestamp"]))

    position = 0
    pnl = 0.0
    pnl_curve = []
    pos_curve = []

    for t in em:
        price = float(t["price"])
        qty = int(t["quantity"])

        # Position-aware quote adjustment
        pos_adj = round(position * 0.15)
        our_bid = FAIR - 1 - pos_adj
        our_ask = FAIR + 1 - pos_adj

        # Incoming buy -> fills our ask
        if price >= our_ask and position > -pos_limit:
            fill = min(qty, pos_limit + position)
            if fill > 0:
                position -= fill
                pnl += our_ask * fill

        # Incoming sell -> fills our bid
        elif price <= our_bid and position < pos_limit:
            fill = min(qty, pos_limit - position)
            if fill > 0:
                position += fill
                pnl -= our_bid * fill

        mtm = pnl + position * FAIR
        pnl_curve.append(mtm)
        pos_curve.append(position)

    return pnl_curve, pos_curve, position, pnl + position * FAIR


def backtest_tomatoes(trades, prices, pos_limit=20):
    """Replay TOMATOES trade flow against EMA market-making strategy."""
    tom_t = sorted(split(trades, "TOMATOES"), key=lambda t: int(t["timestamp"]))
    tom_p = split(prices, "TOMATOES")

    # Pre-compute EMA at each price tick
    alpha_fast = 2 / 21
    alpha_slow = 2 / 101
    ema_f = float(tom_p[0]["mid_price"])
    ema_s = ema_f
    ema_at = {}
    for r in tom_p:
        mid = float(r["mid_price"])
        ema_f = alpha_fast * mid + (1 - alpha_fast) * ema_f
        ema_s = alpha_slow * mid + (1 - alpha_slow) * ema_s
        ema_at[int(r["timestamp"])] = (ema_f, ema_s)

    sorted_ts = sorted(ema_at.keys())
    last_mid = float(tom_p[-1]["mid_price"])

    position = 0
    pnl = 0.0
    pnl_curve = []
    pos_curve = []

    for t in tom_t:
        ts = int(t["timestamp"])
        price = float(t["price"])
        qty = int(t["quantity"])

        # Find nearest EMA
        idx = min(range(len(sorted_ts)), key=lambda i: abs(sorted_ts[i] - ts))
        fv_fast, fv_slow = ema_at[sorted_ts[idx]]
        fair = round(fv_fast)

        # Position + trend adjustments
        pos_adj = round(position * 0.2)
        trend_adj = round((fv_fast - fv_slow) * 0.3)

        half_spread = 5
        our_bid = fair - half_spread - pos_adj + trend_adj
        our_ask = fair + half_spread - pos_adj + trend_adj

        if price >= our_ask and position > -pos_limit:
            fill = min(qty, pos_limit + position)
            if fill > 0:
                position -= fill
                pnl += our_ask * fill

        elif price <= our_bid and position < pos_limit:
            fill = min(qty, pos_limit - position)
            if fill > 0:
                position += fill
                pnl -= our_bid * fill

        mtm = pnl + position * last_mid
        pnl_curve.append(mtm)
        pos_curve.append(position)

    return pnl_curve, pos_curve, position, pnl + position * last_mid


# ── Run backtests ──
results = {}
for label, trades, prices in [("Day -2", t_d2, p_d2), ("Day -1", t_d1, p_d1)]:
    em_pnl, em_pos, em_fpos, em_final = backtest_emeralds(trades)
    tom_pnl, tom_pos, tom_fpos, tom_final = backtest_tomatoes(trades, prices)
    results[label] = {
        "em": (em_pnl, em_pos, em_fpos, em_final),
        "tom": (tom_pnl, tom_pos, tom_fpos, tom_final),
    }
    print(f"{label}:")
    print(f"  EMERALDS  -- final PnL: {em_final:>8,.0f}  final pos: {em_fpos:>3}")
    print(f"  TOMATOES  -- final PnL: {tom_final:>8,.0f}  final pos: {tom_fpos:>3}")
    print(f"  TOTAL     -- {em_final + tom_final:>8,.0f}")
    print()

In [ ]:
# ── PnL & position curves ──
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col, label in enumerate(["Day -2", "Day -1"]):
    em_pnl, em_pos, _, _ = results[label]["em"]
    tom_pnl, tom_pos, _, _ = results[label]["tom"]

    # PnL
    ax = axes[0][col]
    ax.plot(em_pnl, label="EMERALDS", color="steelblue")
    ax.plot(tom_pnl, label="TOMATOES", color="coral")
    total = [e + t for e, t in zip(em_pnl[:min(len(em_pnl), len(tom_pnl))],
                                    tom_pnl[:min(len(em_pnl), len(tom_pnl))])]
    ax.set_title(f"Mark-to-Market PnL -- {label}")
    ax.set_ylabel("PnL")
    ax.legend()
    ax.axhline(0, color="black", lw=0.5)

    # Position
    ax = axes[1][col]
    ax.plot(em_pos, label="EMERALDS", color="steelblue")
    ax.plot(tom_pos, label="TOMATOES", color="coral")
    ax.axhline(0, color="black", lw=0.5)
    ax.axhline(20, color="red", lw=0.5, ls=":")
    ax.axhline(-20, color="red", lw=0.5, ls=":")
    ax.set_title(f"Position -- {label}")
    ax.set_ylabel("Position")
    ax.set_xlabel("Trade #")
    ax.legend()

plt.tight_layout()
plt.show()

### Parameter Sensitivity

How does the strategy PnL change with different quote widths? We sweep the key parameters to confirm our choices are robust, not overfit to one day.

In [ ]:
# ── EMERALDS sensitivity: quote offset from 10000 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax_i, (label, trades) in enumerate([("Day -2", t_d2), ("Day -1", t_d1)]):
    offsets = range(1, 8)
    pnls = []
    for offset in offsets:
        FAIR = 10000
        em = sorted(split(trades, "EMERALDS"), key=lambda t: int(t["timestamp"]))
        position, pnl = 0, 0.0
        for t in em:
            price, qty = float(t["price"]), int(t["quantity"])
            pos_adj = round(position * 0.15)
            our_bid = FAIR - offset - pos_adj
            our_ask = FAIR + offset - pos_adj
            if price >= our_ask and position > -20:
                fill = min(qty, 20 + position)
                if fill > 0:
                    position -= fill
                    pnl += our_ask * fill
            elif price <= our_bid and position < 20:
                fill = min(qty, 20 - position)
                if fill > 0:
                    position += fill
                    pnl -= our_bid * fill
        pnls.append(pnl + position * FAIR)

    axes[ax_i].bar(offsets, pnls, color="steelblue")
    axes[ax_i].set_title(f"EMERALDS -- {label}")
    axes[ax_i].set_xlabel("Quote offset from 10000")
    axes[ax_i].set_ylabel("PnL")
    axes[ax_i].axhline(0, color="black", lw=0.5)

plt.suptitle("EMERALDS: wider quotes = more profit per fill (same fill count until offset > 7)",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── TOMATOES sensitivity: half-spread ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax_i, (label, trades, prices) in enumerate([("Day -2", t_d2, p_d2), ("Day -1", t_d1, p_d1)]):
    spreads = range(2, 9)
    pnls = []
    for hs in spreads:
        _, _, _, final = backtest_tomatoes(trades, prices, pos_limit=20)
        # Re-run with variable half_spread
        tom_t = sorted(split(trades, "TOMATOES"), key=lambda t: int(t["timestamp"]))
        tom_p = split(prices, "TOMATOES")
        af, a_s = 2/21, 2/101
        ef, es = float(tom_p[0]["mid_price"]), float(tom_p[0]["mid_price"])
        ema_at = {}
        for r in tom_p:
            m = float(r["mid_price"])
            ef = af * m + (1 - af) * ef
            es = a_s * m + (1 - a_s) * es
            ema_at[int(r["timestamp"])] = (ef, es)
        sts = sorted(ema_at.keys())
        last_mid = float(tom_p[-1]["mid_price"])

        pos, pnl = 0, 0.0
        for t in tom_t:
            ts, pr, q = int(t["timestamp"]), float(t["price"]), int(t["quantity"])
            idx = min(range(len(sts)), key=lambda i: abs(sts[i] - ts))
            fv_f, fv_s = ema_at[sts[idx]]
            fair = round(fv_f)
            pa = round(pos * 0.2)
            ta = round((fv_f - fv_s) * 0.3)
            ob = fair - hs - pa + ta
            oa = fair + hs - pa + ta
            if pr >= oa and pos > -20:
                f = min(q, 20 + pos)
                if f > 0: pos -= f; pnl += oa * f
            elif pr <= ob and pos < 20:
                f = min(q, 20 - pos)
                if f > 0: pos += f; pnl -= ob * f
        pnls.append(pnl + pos * last_mid)

    axes[ax_i].bar(spreads, pnls, color="coral")
    axes[ax_i].set_title(f"TOMATOES -- {label}")
    axes[ax_i].set_xlabel("Half-spread")
    axes[ax_i].set_ylabel("PnL")
    axes[ax_i].axhline(0, color="black", lw=0.5)

plt.suptitle("TOMATOES: half-spread of 5-6 is the sweet spot", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 5. v1 Summary

| Product | Strategy | Mechanism | Key Parameters |
|---------|----------|-----------|----------------|
| EMERALDS | Market-make at fixed FV | Snipe book + post 9999/10001 | offset=1, pos_lean=0.15 |
| TOMATOES | EMA market-make + trend | Snipe book + post FV +/- 5 | EMA(20/100), spread=5, pos_lean=0.2, trend_wt=0.3 |

---
## 6. Run 98277 Post-Mortem

The v1 strategy scored **522 profit** (EMERALDS 135, TOMATOES 387). Not bad, but the trade-by-trade data exposes two clear problems.

In [ ]:
# Load run 98277 data
with open("run_logs/98277/98277.json") as f:
    run_data = json.load(f)

with open("run_logs/98277/98277.log") as f:
    log_data = json.load(f)

trade_history = log_data["tradeHistory"]
our_trades = [t for t in trade_history if t.get("buyer") == "SUBMISSION" or t.get("seller") == "SUBMISSION"]

# Parse PnL from activities log
act_lines = run_data["activitiesLog"].strip().split("\n")
em_pnl_run, tom_pnl_run, em_ts_run, tom_ts_run = [], [], [], []
for line in act_lines[1:]:
    parts = line.split(";")
    ts, product, pnl = int(parts[1]), parts[2], float(parts[-1])
    if product == "EMERALDS":
        em_pnl_run.append(pnl); em_ts_run.append(ts)
    elif product == "TOMATOES":
        tom_pnl_run.append(pnl); tom_ts_run.append(ts)

# Graph PnL (combined)
graph_lines = run_data["graphLog"].strip().split("\n")
graph_ts = [int(l.split(";")[0]) for l in graph_lines[1:]]
graph_pnl = [float(l.split(";")[1]) for l in graph_lines[1:]]

print(f"Run 98277: total profit = {run_data['profit']:.1f}")
print(f"  EMERALDS final PnL: {em_pnl_run[-1]:.0f}")
print(f"  TOMATOES final PnL: {tom_pnl_run[-1]:.1f}")
print(f"  Max drawdown: {min(graph_pnl):.0f} at t={graph_ts[graph_pnl.index(min(graph_pnl))]}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: PnL over time
axes[0].plot(em_ts_run, em_pnl_run, label="EMERALDS", color="steelblue")
axes[0].plot(tom_ts_run, tom_pnl_run, label="TOMATOES", color="coral")
axes[0].plot(graph_ts, graph_pnl, label="Combined", color="black", lw=1.5)
axes[0].axhline(0, color="gray", lw=0.5)
axes[0].set_title("Run 98277: PnL Over Time")
axes[0].set_xlabel("Timestamp")
axes[0].set_ylabel("PnL")
axes[0].legend(fontsize=8)

# Panel 2: EMERALDS fill price distribution
em_our = [t for t in our_trades if t["symbol"] == "EMERALDS"]
em_buy_prices = [t["price"] for t in em_our if t["buyer"] == "SUBMISSION"]
em_sell_prices = [t["price"] for t in em_our if t["seller"] == "SUBMISSION"]

all_em_prices = sorted(set(em_buy_prices + em_sell_prices))
buy_counts = [sum(1 for p in em_buy_prices if p == ap) for ap in all_em_prices]
sell_counts = [sum(1 for p in em_sell_prices if p == ap) for ap in all_em_prices]

x = np.arange(len(all_em_prices))
axes[1].bar(x - 0.15, buy_counts, 0.3, label="Our buys", color="forestgreen")
axes[1].bar(x + 0.15, sell_counts, 0.3, label="Our sells", color="indianred")
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{int(p)}" for p in all_em_prices], rotation=45)
axes[1].set_title("EMERALDS Fill Prices")
axes[1].set_xlabel("Price")
axes[1].set_ylabel("# Fills")
axes[1].legend(fontsize=8)
axes[1].axvline(x[all_em_prices.index(10000)], color="red", ls="--", lw=1, alpha=0.5)
axes[1].text(x[all_em_prices.index(10000)], max(buy_counts + sell_counts) * 0.9,
             "FV=10000\n(zero edge!)", ha="center", fontsize=8, color="red")

# Panel 3: TOMATOES position over time
tom_our = sorted([t for t in our_trades if t["symbol"] == "TOMATOES"], key=lambda t: t["timestamp"])
tom_pos = []
pos = 0
tom_pos_ts = []
for t in tom_our:
    if t["buyer"] == "SUBMISSION": pos += t["quantity"]
    else: pos -= t["quantity"]
    tom_pos.append(pos)
    tom_pos_ts.append(t["timestamp"])

axes[2].plot(tom_pos_ts, tom_pos, color="coral", marker=".", markersize=3)
axes[2].axhline(0, color="black", lw=0.5)
axes[2].axhline(20, color="red", lw=0.5, ls=":")
axes[2].axhline(-20, color="red", lw=0.5, ls=":")
axes[2].set_title("TOMATOES Position Over Time")
axes[2].set_xlabel("Timestamp")
axes[2].set_ylabel("Position")
axes[2].fill_between(tom_pos_ts, tom_pos, alpha=0.2, color="coral")

plt.tight_layout()
plt.show()

### Diagnosis

**EMERALDS problem: zero-edge sniping at 10000.**
The v1 snipe condition was `price <= FV` (non-strict). When the book tightens and a bot posts at exactly 10000, we aggressively buy/sell there -- gaining zero edge but burning position capacity. **78% of all EMERALDS volume** was at 10000, producing no profit and causing 8 position-limit hits that blocked profitable MM orders.

**TOMATOES problem: EMA(20) lagged a sharp drop.**
Between t=15k and t=45k, the market dropped ~20 points while EMA(20) was still anchored higher. We held long inventory that moved against us, causing a -190 PnL drawdown.

---

## 7. v2 Strategy Changes

| Fix | EMERALDS | TOMATOES |
|-----|----------|----------|
| Snipe threshold | `<` strict (skip 10000) | `< fair-3` (wider buffer) |
| MM spread | +/-3 (was +/-1) | +/-4 (was +/-5) |
| Position lean | 0.25 (was 0.15) | 0.3 (was 0.2) |
| EMA speed | n/a (fixed FV) | EMA(10/50) (was 20/100) |
| FV caps | bid capped at FV-1, ask at FV+1 | -- |
| Trend weight | n/a | 0.4 (was 0.3) |

In [ ]:
# ── v2 backtests ──

def backtest_emeralds_v2(trades, pos_limit=20):
    FAIR = 10000
    em = sorted(split(trades, "EMERALDS"), key=lambda t: int(t["timestamp"]))
    position, pnl = 0, 0.0
    pnl_curve, pos_curve = [], []

    for t in em:
        price, qty = float(t["price"]), int(t["quantity"])

        pos_adj = round(position * 0.25)
        our_bid = min(FAIR - 3 - pos_adj, FAIR - 1)
        our_ask = max(FAIR + 3 - pos_adj, FAIR + 1)

        if price >= our_ask and position > -pos_limit:
            fill = min(qty, pos_limit + position)
            if fill > 0:
                position -= fill; pnl += our_ask * fill
        elif price <= our_bid and position < pos_limit:
            fill = min(qty, pos_limit - position)
            if fill > 0:
                position += fill; pnl -= our_bid * fill

        pnl_curve.append(pnl + position * FAIR)
        pos_curve.append(position)

    return pnl_curve, pos_curve, position, pnl + position * FAIR


def backtest_tomatoes_v2(trades, prices, pos_limit=20):
    tom_t = sorted(split(trades, "TOMATOES"), key=lambda t: int(t["timestamp"]))
    tom_p = split(prices, "TOMATOES")

    alpha_fast, alpha_slow = 2 / 11, 2 / 51
    ema_f = ema_s = float(tom_p[0]["mid_price"])
    ema_at = {}
    for r in tom_p:
        mid = float(r["mid_price"])
        ema_f = alpha_fast * mid + (1 - alpha_fast) * ema_f
        ema_s = alpha_slow * mid + (1 - alpha_slow) * ema_s
        ema_at[int(r["timestamp"])] = (ema_f, ema_s)

    sorted_ts = sorted(ema_at.keys())
    last_mid = float(tom_p[-1]["mid_price"])

    position, pnl = 0, 0.0
    pnl_curve, pos_curve = [], []

    for t in tom_t:
        ts, price, qty = int(t["timestamp"]), float(t["price"]), int(t["quantity"])
        idx = min(range(len(sorted_ts)), key=lambda i: abs(sorted_ts[i] - ts))
        fv_f, fv_s = ema_at[sorted_ts[idx]]
        fair = round(fv_f)

        pos_adj = round(position * 0.3)
        trend_adj = round((fv_f - fv_s) * 0.4)
        our_bid = fair - 4 - pos_adj + trend_adj
        our_ask = fair + 4 - pos_adj + trend_adj

        if price >= our_ask and position > -pos_limit:
            fill = min(qty, pos_limit + position)
            if fill > 0:
                position -= fill; pnl += our_ask * fill
        elif price <= our_bid and position < pos_limit:
            fill = min(qty, pos_limit - position)
            if fill > 0:
                position += fill; pnl -= our_bid * fill

        pnl_curve.append(pnl + position * last_mid)
        pos_curve.append(position)

    return pnl_curve, pos_curve, position, pnl + position * last_mid


# Compare v1 vs v2
print(f"{'':>8} {'EMERALDS v1':>14} {'EMERALDS v2':>14} {'TOMATOES v1':>14} {'TOMATOES v2':>14} {'Total v1':>10} {'Total v2':>10}")
print("-" * 90)
for label, trades, prices in [("Day -2", t_d2, p_d2), ("Day -1", t_d1, p_d1)]:
    _, _, _, em1 = backtest_emeralds(trades)
    _, _, _, em2 = backtest_emeralds_v2(trades)
    _, _, _, tm1 = backtest_tomatoes(trades, prices)
    _, _, _, tm2 = backtest_tomatoes_v2(trades, prices)
    print(f"{label:>8} {em1:>14,.0f} {em2:>14,.0f} {tm1:>14,.0f} {tm2:>14,.0f} {em1+tm1:>10,.0f} {em2+tm2:>10,.0f}")

In [ ]:
# ── v1 vs v2 PnL curves side by side ──
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col, (label, trades, prices) in enumerate([("Day -2", t_d2, p_d2), ("Day -1", t_d1, p_d1)]):
    em1_pnl, em1_pos, _, _ = backtest_emeralds(trades)
    em2_pnl, em2_pos, _, _ = backtest_emeralds_v2(trades)
    tm1_pnl, tm1_pos, _, _ = backtest_tomatoes(trades, prices)
    tm2_pnl, tm2_pos, _, _ = backtest_tomatoes_v2(trades, prices)

    # PnL comparison
    ax = axes[0][col]
    ax.plot(em1_pnl, label="EMERALDS v1", color="steelblue", alpha=0.5, ls="--")
    ax.plot(em2_pnl, label="EMERALDS v2", color="steelblue", lw=2)
    ax.plot(tm1_pnl, label="TOMATOES v1", color="coral", alpha=0.5, ls="--")
    ax.plot(tm2_pnl, label="TOMATOES v2", color="coral", lw=2)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(f"PnL: v1 (dashed) vs v2 (solid) -- {label}")
    ax.set_ylabel("PnL")
    ax.legend(fontsize=7)

    # Position comparison
    ax = axes[1][col]
    ax.plot(em1_pos, label="EMERALDS v1", color="steelblue", alpha=0.5, ls="--")
    ax.plot(em2_pos, label="EMERALDS v2", color="steelblue", lw=2)
    ax.plot(tm1_pos, label="TOMATOES v1", color="coral", alpha=0.5, ls="--")
    ax.plot(tm2_pos, label="TOMATOES v2", color="coral", lw=2)
    ax.axhline(0, color="black", lw=0.5)
    ax.axhline(20, color="red", lw=0.5, ls=":")
    ax.axhline(-20, color="red", lw=0.5, ls=":")
    ax.set_title(f"Position -- {label}")
    ax.set_ylabel("Position")
    ax.set_xlabel("Trade #")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()